# Geospatial Data Retrieval Agent: Single-Agent Examples

This notebook shows how to call the GAS `geospatial_data_retrieval_agent` directly with the Python `gas-client` SDK.

It is intentionally self-contained:

- install `gas-client`
- configure credentials and a GAS server URL
- inspect the agent's `DescribeAgent` JSON
- run `ExecuteTask` in `stream`, `async`, and `sync` modes, in that order

The data requests below use public data sources that do not require source-specific API keys. The model-backed agent still needs an LLM credential accepted by the server. Check the agent's `DescribeAgent` JSON to confirm which credential names your deployment accepts.

## 1. Install The GAS Client SDK

Run this cell once in a fresh notebook environment.

In [ ]:
%pip install -q "gas-client>=0.2.1"

## 2. Imports And Helper Functions

In [ ]:
import json
import os
from pathlib import Path
from urllib.parse import urljoin, urlparse

import requests
from IPython.display import HTML, Image, display

from gas_client import GasClient


def pretty(data):
    print(json.dumps(data, indent=2, ensure_ascii=False))


def artifact_display_name(artifact):
    return artifact.get("filename") or artifact.get("name") or artifact.get("url") or "artifact"


def artifact_url(artifact, server_url):
    url = artifact.get("url")
    if not url:
        return None
    if url.startswith("/"):
        return urljoin(server_url.rstrip("/") + "/", url)
    return url


def first_artifact_url(task_result, server_url, preferred_extensions=None):
    artifacts = task_result.get("outputs", {}).get("artifacts", [])
    preferred_extensions = preferred_extensions or []

    for extension in preferred_extensions:
        for artifact in artifacts:
            url = artifact_url(artifact, server_url)
            name = artifact_display_name(artifact)
            if url and str(name).lower().endswith(extension.lower()):
                return url

    for artifact in artifacts:
        url = artifact_url(artifact, server_url)
        if url:
            return url
    return None


def display_artifact_links(task_result, server_url):
    artifacts = task_result.get("outputs", {}).get("artifacts", [])
    if not artifacts:
        print("No artifacts returned.")
        return

    rows = []
    for index, artifact in enumerate(artifacts, start=1):
        url = artifact_url(artifact, server_url)
        name = artifact_display_name(artifact)
        fmt = artifact.get("format") or Path(urlparse(url or name).path).suffix.lstrip(".") or ""
        size = artifact.get("size") or artifact.get("size_bytes") or ""
        if url:
            link = f'<a href="{url}" target="_blank" rel="noopener">{name}</a>'
        else:
            link = name
        rows.append(f"<tr><td>{index}</td><td>{link}</td><td>{fmt}</td><td>{size}</td></tr>")

    html = (
        "<table>"
        "<thead><tr><th>#</th><th>Artifact</th><th>Format</th><th>Size</th></tr></thead>"
        f"<tbody>{''.join(rows)}</tbody>"
        "</table>"
    )
    display(HTML(html))


def absolute_url(url):
    if not url:
        return None
    if url.startswith("/"):
        return urljoin(server_url.rstrip("/") + "/", url)
    return url


def local_artifact_path(artifact, url):
    cache_dir = Path("downloaded_gas_artifacts")
    cache_dir.mkdir(parents=True, exist_ok=True)

    filename = artifact_display_name(artifact)
    if not Path(filename).suffix:
        filename = Path(urlparse(url).path).name or "gas_artifact"
    return cache_dir / filename


def download_artifact(artifact):
    url = absolute_url(artifact.get("url"))
    if not url:
        raise ValueError("Artifact does not include a URL.")
    path = local_artifact_path(artifact, url)

    if not path.exists():
        response = requests.get(url, timeout=300)
        response.raise_for_status()
        path.write_bytes(response.content)

    return path


def display_task_artifacts(task_result, max_vector_rows=5):
    """Display common GAS artifacts inline when possible."""

    artifacts = task_result.get("outputs", {}).get("artifacts", [])
    if not artifacts:
        print("No artifacts to display.")
        return

    for artifact in artifacts:
        url = artifact.get("url")
        if not url:
            continue

        name = artifact_display_name(artifact)
        suffix = Path(name).suffix.lower() or Path(urlparse(url).path).suffix.lower()
        print(f"Artifact: {name}")

        try:
            path = download_artifact(artifact)

            if suffix in {".geojson", ".json", ".gpkg", ".shp"}:
                import geopandas as gpd
                import matplotlib.pyplot as plt

                gdf = gpd.read_file(path)
                ax = gdf.plot(figsize=(10, 6), edgecolor="black", linewidth=0.3, alpha=0.8)
                ax.set_axis_off()
                display(plt.gcf())
                plt.close()
                display(gdf.head(max_vector_rows))
            elif suffix == ".csv":
                import pandas as pd

                display(pd.read_csv(path).head(max_vector_rows))
            elif suffix in {".png", ".jpg", ".jpeg", ".gif"}:
                display(Image(filename=str(path)))
            elif suffix in {".html", ".htm"}:
                html = path.read_text(encoding="utf-8", errors="ignore")
                display(HTML(f'<iframe srcdoc={html!r} width="100%" height="720" style="border:1px solid #ddd;"></iframe>'))
            else:
                display(HTML(f'<a href="{absolute_url(url)}" target="_blank">Open or download artifact</a>'))
        except Exception as exc:
            print(f"Could not render this artifact inline: {exc}")
            display(HTML(f'<a href="{absolute_url(url)}" target="_blank">Open or download artifact</a>'))


def final_result_from_stream(events):
    for event in reversed(events):
        if event.get("event") == "task_result" and isinstance(event.get("payload"), dict):
            return event["payload"]
        if event.get("type") == "task_result" and isinstance(event.get("payload"), dict):
            return event["payload"]
    return None


## 3. Configure Server And Credentials

Use `default_credentials` for credentials that should be sent with every task request. You can override or add request-specific credentials in an individual `execute_task(...)` call.

The public GIBD deployment currently accepts `OPENAI_API_KEY` or `GIBD_API_KEY` for this model-backed agent. Other deployments may use different names, so check `DescribeAgent` before deciding which key to send.

In [ ]:
server_url = os.getenv("GAS_SERVER_URL", "https://www.geospatial-agentic-services.online")

# Choose one credential accepted by your server/agent. Leave the other unset.
default_credentials = {}
openai_api_key = os.getenv("OPENAI_API_KEY")
gibd_api_key = os.getenv("GIBD_API_KEY")

if openai_api_key:
    default_credentials["OPENAI_API_KEY"] = openai_api_key
if gibd_api_key:
    default_credentials["GIBD_API_KEY"] = gibd_api_key

# For quick local experimentation, you may paste a temporary value here instead.
# default_credentials["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"

client = GasClient(
    server_url,
    default_credentials=default_credentials,
    artifact_delivery="URL",
    timeout=900,
)

agent = client.agent("geospatial_data_retrieval_agent")
print("Server:", server_url)
print("Default credential names:", sorted(default_credentials) or "none configured")


## 4. Inspect DescribeAgent

Use `DescribeAgent` to confirm supported modes, credentials, parameters, and source-specific credential requirements.

In [ ]:
describe_agent = agent.describe()

profile = describe_agent.get("profile", {})
execute_task = describe_agent.get("execute_task", {})
print("Agent:", profile.get("name"))
print("Agent ID:", profile.get("agent_id"))
print("Supported modes:", execute_task.get("modes"))
print("Accepted model credential names:", execute_task.get("credentials", {}).get("one_of"))

# Uncomment to inspect the full profile.
# pretty(describe_agent)


## 5. Stream Mode

`stream` mode keeps the connection open and yields progress events while the task runs. The final `task_result` event contains the standard GAS task response.

In [ ]:
stream_instructions = (
    "Download Pennsylvania county boundaries from the US Census Bureau. "
    "Return a geospatial vector dataset with county names and identifiers."
)

stream_events = []
stream_result = None

for event in agent.execute_task(
    stream_instructions,
    mode="stream",
    artifact_delivery="URL",
    timeout=900,
):
    stream_events.append(event)
    client.print_stream_event(event)

stream_result = final_result_from_stream(stream_events)
if stream_result is None:
    raise RuntimeError("The stream ended before returning a task_result event.")

client.print_task_summary(stream_result)
display_artifact_links(stream_result, server_url)
display_task_artifacts(stream_result)


## 6. Stream Mode: Census Demographic Data

This second streaming example retrieves Census demographic attributes. It demonstrates a data-source credential override through `credentials.source_credentials` while still using the client-level default LLM credential.

In [ ]:
census_stream_instructions = (
    "Download 2021 county-level total population data for Pennsylvania from the US Census Bureau. "
    "Return a dataset with county names, GEOID identifiers, and total population values."
)

census_credentials = {
    "source_credentials": {
        "US_Census_demography": {
            "key": us_census_demography_key,
        }
    }
} if "us_census_demography_key" in globals() and us_census_demography_key else None

census_stream_events = []
census_stream_result = None

for event in agent.execute_task(
    census_stream_instructions,
    mode="stream",
    artifact_delivery="URL",
    credentials=census_credentials,
    timeout=900,
):
    census_stream_events.append(event)
    client.print_stream_event(event)

census_stream_result = final_result_from_stream(census_stream_events)
if census_stream_result is None:
    raise RuntimeError("The stream ended before returning a task_result event.")

client.print_task_summary(census_stream_result)
display_artifact_links(census_stream_result, server_url)
display_task_artifacts(census_stream_result)


## 7. Async Mode

`async` mode returns quickly with a task ID. The client then polls until the task reaches a terminal status and retrieves the final result.

In [ ]:
async_instructions = (
    "Download USGS earthquake events with magnitude 4.5 or greater from the past 30 days. "
    "Return the results as a geospatial point dataset."
)

async_submission = agent.execute_task(
    async_instructions,
    mode="async",
    artifact_delivery="URL",
    timeout=60,
)

pretty(async_submission)
async_task_id = client.get_task_id(async_submission)
print("Async task ID:", async_task_id)

async_result = agent.wait_for_task(
    async_task_id,
    poll_interval=5,
    timeout_seconds=900,
)

client.print_task_summary(async_result)
display_artifact_links(async_result, server_url)
display_task_artifacts(async_result)


## 8. Sync Mode

`sync` mode waits for the final task result in the original request. It is simplest for short or moderate tasks.

In [ ]:
sync_instructions = (
    "Download CDC PLACES county-level obesity prevalence data for Pennsylvania. "
    "Return a tabular or geospatial dataset with county identifiers and obesity measure values."
)

sync_result = agent.execute_task(
    sync_instructions,
    mode="sync",
    artifact_delivery="URL",
    timeout=900,
)

client.print_task_summary(sync_result)
display_artifact_links(sync_result, server_url)
display_task_artifacts(sync_result)


## 9. Compare The Returned Artifacts

Each execution mode returns the same standard GAS task response shape. The main difference is how you receive the final response.

In [ ]:
results = {
    "stream": stream_result,
    "census_stream": census_stream_result,
    "async": async_result,
    "sync": sync_result,
}

for mode, result in results.items():
    status = result.get("task", {}).get("status")
    artifact_count = len(result.get("outputs", {}).get("artifacts", []))
    first_url = first_artifact_url(result, server_url)
    print(f"{mode:>6}: status={status}, artifacts={artifact_count}, first_artifact={first_url}")
